# ETL — Minnie65: Cell-Cell Connectivity (Long)

Writes `CellCellConnectivityLong` synapse data for Minnie65 v1412 using two parallel examples:

1. **Example 1** — precomputed parquet: (proofread ∩ CSM)-pre × CSM-post, two measurement types (`SYNAPSE_COUNT`, `SUM_ANATOMICAL_SIZE`).
2. **Example 2** — CAVE `connections_with_nuclei` view: (proofread ∩ CSM)-pre × (proofread ∩ CSM)-post, `SYNAPSE_COUNT` only (best-effort; skips on failure).

Also creates the `minnie65_v1412_proofread` cohort DataSet (cells that are both proofread AND in the CSM cohort) and its associations.

**Outputs:**
- `dataset/` — +1 row (`minnie65_v1412_proofread`)
- `dataitem_dataset_association/` — one per proofread ∩ CSM cell
- `cellcellconnectivitylong_proofread_pre_to_csm_post/` — 2 rows per pair
- `cellcellconnectivitylong_proofread_to_proofread/` — 1 row per pair (if CAVE view succeeds)

**Prerequisites:** `etl_minnie_01_dataset_dataitem.ipynb`, `etl_minnie_02_cell_features.ipynb`.

Identifiers: `project_id = "minnie65"`, `dataset_id = "minnie65_v1412_proofread"`.

In [1]:
import os

import caveclient
import pandas as pd
import polars as pl
import pyarrow as pa
from deltalake import write_deltalake

from connects_common_connectivity.arrow_utils import (
    build_arrow_schema,
    models_to_table,
    attach_linkml_metadata,
)
from connects_common_connectivity.models import (
    CellCellConnectivityLong,
    DataSet,
    DataItemDataSetAssociation,
    Modality,
    SynapticMeasurementType,
    Unit,
)

/opt/conda/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.2.2) or chardet (7.4.3)/charset_normalizer (3.3.2) doesn't match a supported version!
  warnings.warn(


In [2]:
OUTPUT_ROOT = "../scratch/em_patchseq_wnm_v1/"
PROJECT_ID = "minnie65"
DATASET_ID_CSM = "minnie65_v1412_csm_cluster"
DATASET_ID_PROOFREAD = "minnie65_v1412_proofread"
PARQUET_PATH = "/data/minnie1412/minnie_soma_soma_connectivity.parquet"
CAVE_DATASTACK = "minnie65_phase3_v1"
CAVE_VERSION = 1412
CAVE_NUC_VIEW = "nucleus_detection_lookup_v1"
CAVE_PROOFREAD_TABLE = "proofreading_status_and_strategy"
CAVE_CONN_VIEW = "connections_with_nuclei"

print(f"OUTPUT_ROOT            : {OUTPUT_ROOT}")
print(f"PROJECT_ID             : {PROJECT_ID}")
print(f"DATASET_ID_CSM         : {DATASET_ID_CSM}")
print(f"DATASET_ID_PROOFREAD   : {DATASET_ID_PROOFREAD}")
print(f"PARQUET_PATH           : {PARQUET_PATH}")
print(f"CAVE_DATASTACK         : {CAVE_DATASTACK}")
print(f"CAVE_VERSION           : {CAVE_VERSION}")
print(f"CAVE_NUC_VIEW          : {CAVE_NUC_VIEW}")
print(f"CAVE_PROOFREAD_TABLE   : {CAVE_PROOFREAD_TABLE}")
print(f"CAVE_CONN_VIEW         : {CAVE_CONN_VIEW}")

OUTPUT_ROOT            : ../scratch/em_patchseq_wnm_v1/
PROJECT_ID             : minnie65
DATASET_ID_CSM         : minnie65_v1412_csm_cluster
DATASET_ID_PROOFREAD   : minnie65_v1412_proofread
PARQUET_PATH           : /data/minnie1412/minnie_soma_soma_connectivity.parquet
CAVE_DATASTACK         : minnie65_phase3_v1
CAVE_VERSION           : 1412
CAVE_NUC_VIEW          : nucleus_detection_lookup_v1
CAVE_PROOFREAD_TABLE   : proofreading_status_and_strategy
CAVE_CONN_VIEW         : connections_with_nuclei


In [3]:
# Prereq: CSM cohort associations must exist (written by etl_minnie_02)
csm_assoc = pl.read_delta(OUTPUT_ROOT + "dataitem_dataset_association/").filter(
    (pl.col("project_id") == PROJECT_ID) & (pl.col("dataset_id") == DATASET_ID_CSM)
)
assert csm_assoc.shape[0] > 0, (
    "No CSM cohort associations found. Run etl_minnie_02_cell_features.ipynb first."
)
csm_nuc_ids = set(csm_assoc["dataitem_id"].to_list())
print(f"CSM cohort cells loaded: {len(csm_nuc_ids)}")

CSM cohort cells loaded: 35783


---
## Build proofread ∩ CSM cohort

Query CAVE `proofreading_status_and_strategy` table (filter `status_axon`), join to `nucleus_detection_lookup_v1` to get nucleus ids, then intersect with the CSM cohort. The resulting set defines the `minnie65_v1412_proofread` DataSet.

In [4]:
client = caveclient.CAVEclient(CAVE_DATASTACK, auth_token=os.environ["CUSTOM_KEY"])
client.materialize.version = CAVE_VERSION

# Get proofread cells (status_axon == True)
proof_df = client.materialize.query_table(
    CAVE_PROOFREAD_TABLE, materialization_version=CAVE_VERSION
)
proof_df = proof_df.query("status_axon == True")
print(f"Proofread cells (status_axon): {proof_df.shape[0]}")

# Get nucleus lookup to map pt_root_id → nucleus id
nuc_df = client.materialize.query_view(CAVE_NUC_VIEW)
nuc_df = nuc_df.query("pt_root_id != 0")
print(f"Nucleus lookup rows: {nuc_df.shape[0]}")

# Join: proofread pt_root_id → nucleus id
proof_with_nuc = proof_df.merge(
    nuc_df[["id", "pt_root_id"]].rename(columns={"id": "nuc_id"}),
    on="pt_root_id",
    how="inner",
)
all_proofread_nuc_ids = set(proof_with_nuc["nuc_id"].astype(str).tolist())
print(f"Proofread cells with nucleus id: {len(all_proofread_nuc_ids)}")

Proofread cells (status_axon): 1982
Nucleus lookup rows: 133969
Proofread cells with nucleus id: 1962


In [5]:
# The proofread cohort DataSet is defined as proofread ∩ CSM
proofread_nuc_ids = all_proofread_nuc_ids & csm_nuc_ids
proof_only = all_proofread_nuc_ids - csm_nuc_ids

print(f"|all proofread|        : {len(all_proofread_nuc_ids)}")
print(f"|csm|                  : {len(csm_nuc_ids)}")
print(f"|proofread ∩ csm|      : {len(proofread_nuc_ids)}  ← this is the proofread DataSet")
print(f"|proofread \\ csm|      : {len(proof_only)}")

|all proofread|        : 1962
|csm|                  : 35783
|proofread ∩ csm|      : 1861  ← this is the proofread DataSet
|proofread \ csm|      : 101


The `minnie65_v1412_proofread` DataSet contains cells that have both verified axon reconstructions AND CSM dendrite classifications. These cells form the pre-synaptic population for both examples. Example 1 uses all CSM cells as the post-synaptic population; Example 2 restricts both sides to the proofread ∩ CSM set.

In [6]:
# All proofread ∩ CSM cells must already be DataItems (registered by etl_minnie_01)
existing_items = set(
    pl.read_delta(OUTPUT_ROOT + "dataitem/")
    .filter(pl.col("project_id") == PROJECT_ID)["id"]
    .to_list()
)
missing = proofread_nuc_ids - existing_items
assert len(missing) == 0, (
    f"{len(missing)} proofread ∩ CSM nucleus ids not found in dataitem/. "
    "Run etl_minnie_01_dataset_dataitem.ipynb first."
)
print(f"All {len(proofread_nuc_ids)} proofread ∩ CSM cells confirmed in dataitem/.")

All 1861 proofread ∩ CSM cells confirmed in dataitem/.


In [7]:
proofread_ds = DataSet(
    id=DATASET_ID_PROOFREAD,
    name="Minnie65 v1412 proofread axon cohort with CSM features",
    modality=Modality.ELECTRON_MICROSCOPY.value,
    project_id=PROJECT_ID,
)

schema_ds = build_arrow_schema(DataSet)
table_ds = attach_linkml_metadata(
    models_to_table([proofread_ds], schema=schema_ds), linkml_class="DataSet"
)

write_deltalake(
    OUTPUT_ROOT + "dataset/",
    table_ds,
    mode="overwrite",
    predicate=f"project_id = '{PROJECT_ID}' AND id = '{DATASET_ID_PROOFREAD}'",
    partition_by=["project_id"],
)
print(f"DataSet written: {table_ds.num_rows} row(s)")

DataSet written: 1 row(s)


In [8]:
ds_v = pl.read_delta(OUTPUT_ROOT + "dataset/").filter(
    (pl.col("project_id") == PROJECT_ID) & (pl.col("id") == DATASET_ID_PROOFREAD)
)
print(ds_v.shape)
print(ds_v.head(3))
assert ds_v.shape[0] == 1

(1, 5)
shape: (1, 5)
┌──────────────────────────┬──────────────────┬─────────────┬─────────────────────┬────────────┐
│ id                       ┆ name             ┆ publication ┆ modality            ┆ project_id │
│ ---                      ┆ ---              ┆ ---         ┆ ---                 ┆ ---        │
│ str                      ┆ str              ┆ str         ┆ str                 ┆ str        │
╞══════════════════════════╪══════════════════╪═════════════╪═════════════════════╪════════════╡
│ minnie65_v1412_proofread ┆ Minnie65 v1412   ┆ null        ┆ ELECTRON_MICROSCOPY ┆ minnie65   │
│                          ┆ proofread axon … ┆             ┆                     ┆            │
└──────────────────────────┴──────────────────┴─────────────┴─────────────────────┴────────────┘


In [9]:
associations = [
    DataItemDataSetAssociation(
        dataitem_id=nuc_id, dataset_id=DATASET_ID_PROOFREAD, project_id=PROJECT_ID
    )
    for nuc_id in sorted(proofread_nuc_ids)
]

schema_assoc = build_arrow_schema(DataItemDataSetAssociation)
table_assoc = attach_linkml_metadata(
    models_to_table(associations, schema=schema_assoc),
    linkml_class="DataItemDataSetAssociation",
)

write_deltalake(
    OUTPUT_ROOT + "dataitem_dataset_association/",
    table_assoc,
    mode="overwrite",
    predicate=f"project_id = '{PROJECT_ID}' AND dataset_id = '{DATASET_ID_PROOFREAD}'",
    partition_by=["project_id"],
)
print(f"DataItemDataSetAssociation written: {table_assoc.num_rows} row(s) (proofread ∩ CSM)")

DataItemDataSetAssociation written: 1861 row(s) (proofread ∩ CSM)


In [10]:
assoc_v = pl.read_delta(OUTPUT_ROOT + "dataitem_dataset_association/").filter(
    (pl.col("project_id") == PROJECT_ID) & (pl.col("dataset_id") == DATASET_ID_PROOFREAD)
)
print(assoc_v.shape)
print(assoc_v.head(3))
assert assoc_v.shape[0] == len(proofread_nuc_ids)

(1861, 3)
shape: (3, 3)
┌─────────────┬──────────────────────────┬────────────┐
│ dataitem_id ┆ dataset_id               ┆ project_id │
│ ---         ┆ ---                      ┆ ---        │
│ str         ┆ str                      ┆ str        │
╞═════════════╪══════════════════════════╪════════════╡
│ 188569      ┆ minnie65_v1412_proofread ┆ minnie65   │
│ 188961      ┆ minnie65_v1412_proofread ┆ minnie65   │
│ 189149      ┆ minnie65_v1412_proofread ┆ minnie65   │
└─────────────┴──────────────────────────┴────────────┘


---
## Example 1 — Precomputed parquet: (proofread ∩ CSM)-pre × CSM-post

Load the soma-to-soma connectivity parquet, filter to pre-synaptic cells in the proofread ∩ CSM set and post-synaptic cells in the full CSM set, then write two `CellCellConnectivityLong` measurement types per pair.

In [11]:
conn_df = pl.read_parquet(PARQUET_PATH)
print(f"Raw parquet rows: {conn_df.shape[0]}")

# Cast nuc_id columns to string for matching
conn_df = conn_df.with_columns(
    pl.col("pre_nuc_id").cast(pl.Utf8).alias("pre_nuc_id_str"),
    pl.col("post_nuc_id").cast(pl.Utf8).alias("post_nuc_id_str"),
)

# Filter: pre in proofread ∩ CSM, post in CSM
conn_filtered = conn_df.filter(
    pl.col("pre_nuc_id_str").is_in(proofread_nuc_ids)
    & pl.col("post_nuc_id_str").is_in(csm_nuc_ids)
)
print(f"Filtered rows ((proofread ∩ CSM)-pre × CSM-post): {conn_filtered.shape[0]}")
conn_filtered.head(3)

Raw parquet rows: 796098
Filtered rows ((proofread ∩ CSM)-pre × CSM-post): 627757


pre_pt_root_id,post_pt_root_id,n_syn,sum_size,pre_nuc_id,post_nuc_id,__index_level_0__,pre_nuc_id_str,post_nuc_id_str
i64,i64,i64,i64,i64,i64,i64,str,str
864691135373601736,864691134884759546,2,6244,273595,205051,43183978,"""273595""","""205051"""
864691135688823008,864691134884759546,1,5460,271561,205051,43185183,"""271561""","""205051"""
864691135684284914,864691134884759546,1,6636,275879,205051,43185217,"""275879""","""205051"""


In [12]:
pre_ids = set(conn_filtered["pre_nuc_id_str"].to_list())
post_ids = set(conn_filtered["post_nuc_id_str"].to_list())
all_conn_ids = pre_ids | post_ids

missing_items = all_conn_ids - existing_items
assert len(missing_items) == 0, (
    f"{len(missing_items)} cell ids in connectivity data not found in dataitem/."
)
print(f"All {len(all_conn_ids)} unique cell ids confirmed in dataitem/.")

All 34611 unique cell ids confirmed in dataitem/.


In [13]:
rows_ex1 = []
for row in conn_filtered.iter_rows(named=True):
    pre = row["pre_nuc_id_str"]
    post = row["post_nuc_id_str"]
    rows_ex1.append(
        CellCellConnectivityLong(
            id=f"{pre}_{post}_{SynapticMeasurementType.SYNAPSE_COUNT.value}",
            presynaptic_cell=pre,
            postsynaptic_cell=post,
            measurement_type=SynapticMeasurementType.SYNAPSE_COUNT.value,
            modality=Modality.ELECTRON_MICROSCOPY.value,
            value=float(row["n_syn"]),
            unit=Unit.COUNT.value,
            project_id=PROJECT_ID,
        )
    )
    rows_ex1.append(
        CellCellConnectivityLong(
            id=f"{pre}_{post}_{SynapticMeasurementType.SUM_ANATOMICAL_SIZE.value}",
            presynaptic_cell=pre,
            postsynaptic_cell=post,
            measurement_type=SynapticMeasurementType.SUM_ANATOMICAL_SIZE.value,
            modality=Modality.ELECTRON_MICROSCOPY.value,
            value=float(row["sum_size"]),
            unit=Unit.ARBITRARY_UNIT.value,
            project_id=PROJECT_ID,
        )
    )

print(f"CellCellConnectivityLong rows (Example 1): {len(rows_ex1)}")

CellCellConnectivityLong rows (Example 1): 1255514


In [14]:
schema_cc = build_arrow_schema(CellCellConnectivityLong)
table_ex1 = attach_linkml_metadata(
    models_to_table(rows_ex1, schema=schema_cc),
    linkml_class="CellCellConnectivityLong",
)

write_deltalake(
    OUTPUT_ROOT + "cellcellconnectivitylong_proofread_pre_to_csm_post/",
    table_ex1,
    mode="overwrite",
    predicate=f"project_id = '{PROJECT_ID}'",
    partition_by=["project_id", "measurement_type"],
)
print(f"Written to cellcellconnectivitylong_proofread_pre_to_csm_post/: {table_ex1.num_rows} rows")

Written to cellcellconnectivitylong_proofread_pre_to_csm_post/: 1255514 rows


In [15]:
ex1_v = pl.read_delta(OUTPUT_ROOT + "cellcellconnectivitylong_proofread_pre_to_csm_post/").filter(
    pl.col("project_id") == PROJECT_ID
)
print(f"Shape: {ex1_v.shape}")
print(ex1_v.group_by("measurement_type").len())
print(ex1_v.head(3))
assert ex1_v.shape[0] == len(rows_ex1)
assert ex1_v["measurement_type"].n_unique() == 2, "Expected both SYNAPSE_COUNT and SUM_ANATOMICAL_SIZE"

Shape: (1255514, 9)
shape: (2, 2)
┌─────────────────────┬────────┐
│ measurement_type    ┆ len    │
│ ---                 ┆ ---    │
│ str                 ┆ u32    │
╞═════════════════════╪════════╡
│ SUM_ANATOMICAL_SIZE ┆ 627757 │
│ SYNAPSE_COUNT       ┆ 627757 │
└─────────────────────┴────────┘
shape: (3, 9)
┌────────────┬───────────┬───────────┬───────────┬───┬─────────┬───────────┬───────────┬───────────┐
│ id         ┆ descripti ┆ presynapt ┆ postsynap ┆ … ┆ value   ┆ unit      ┆ project_i ┆ measureme │
│ ---        ┆ on        ┆ ic_cell   ┆ tic_cell  ┆   ┆ ---     ┆ ---       ┆ d         ┆ nt_type   │
│ str        ┆ ---       ┆ ---       ┆ ---       ┆   ┆ f64     ┆ str       ┆ ---       ┆ ---       │
│            ┆ str       ┆ str       ┆ str       ┆   ┆         ┆           ┆ str       ┆ str       │
╞════════════╪═══════════╪═══════════╪═══════════╪═══╪═════════╪═══════════╪═══════════╪═══════════╡
│ 292713_374 ┆ null      ┆ 292713    ┆ 374158    ┆ … ┆ 920.0   ┆ ARBITRARY ┆ minni

---
## Example 2 — CAVE `connections_with_nuclei` view: (proofread ∩ CSM) × (proofread ∩ CSM) (TAKES TOO LONG 30+mins)

Queries the CAVE view for pre-aggregated synapse counts. Both pre and post are restricted to the proofread ∩ CSM set. If the view is unavailable, this section is skipped with a warning.

In [16]:
example2_success = False
conn_view_df = None

try:
    conn_view_df = client.materialize.query_view(
        CAVE_CONN_VIEW, materialization_version=CAVE_VERSION
    )
    print(f"CAVE view '{CAVE_CONN_VIEW}' loaded: {conn_view_df.shape}")
    print(f"Columns: {conn_view_df.columns.tolist()}")
    conn_view_df.head(3)
except Exception as e:
    print(f"⚠️  CAVE view '{CAVE_CONN_VIEW}' unavailable: {e}")
    print("Skipping Example 2 writes.")

KeyboardInterrupt: 

In [ ]:
if conn_view_df is not None:
    # Identify the synapse count column
    count_col_candidates = [c for c in conn_view_df.columns if "syn" in c.lower() and "count" in c.lower()]
    if not count_col_candidates:
        count_col_candidates = [c for c in conn_view_df.columns if c in ("n_syn", "synapse_count", "syn_count")]
    if not count_col_candidates:
        count_col_candidates = [c for c in conn_view_df.columns if "n_syn" in c]

    if not count_col_candidates:
        print(f"⚠️  Cannot identify synapse count column from: {conn_view_df.columns.tolist()}")
        print("Skipping Example 2 writes.")
    else:
        count_col = count_col_candidates[0]
        print(f"Using synapse count column: '{count_col}'")

        # Identify pre/post nucleus id columns
        pre_nuc_col_candidates = [c for c in conn_view_df.columns if "pre" in c.lower() and "nuc" in c.lower()]
        post_nuc_col_candidates = [c for c in conn_view_df.columns if "post" in c.lower() and "nuc" in c.lower()]

        if not pre_nuc_col_candidates or not post_nuc_col_candidates:
            print(f"⚠️  Cannot identify pre/post nuc id columns from: {conn_view_df.columns.tolist()}")
            print("Skipping Example 2 writes.")
        else:
            pre_nuc_col = pre_nuc_col_candidates[0]
            post_nuc_col = post_nuc_col_candidates[0]
            print(f"Pre nuc col: '{pre_nuc_col}', Post nuc col: '{post_nuc_col}'")

            # Convert to polars for filtering
            conn_view_pl = pl.from_pandas(conn_view_df)
            conn_view_pl = conn_view_pl.with_columns(
                pl.col(pre_nuc_col).cast(pl.Utf8).alias("pre_nuc_str"),
                pl.col(post_nuc_col).cast(pl.Utf8).alias("post_nuc_str"),
            )

            # Filter: both pre and post in proofread ∩ CSM
            conn_view_filtered = conn_view_pl.filter(
                pl.col("pre_nuc_str").is_in(proofread_nuc_ids)
                & pl.col("post_nuc_str").is_in(proofread_nuc_ids)
            )
            print(f"Filtered rows ((proofread ∩ CSM) × (proofread ∩ CSM)): {conn_view_filtered.shape[0]}")

            if conn_view_filtered.shape[0] == 0:
                print("⚠️  No rows after filtering. Skipping Example 2 writes.")
            else:
                # Assert all ids are DataItems
                ex2_pre_ids = set(conn_view_filtered["pre_nuc_str"].to_list())
                ex2_post_ids = set(conn_view_filtered["post_nuc_str"].to_list())
                ex2_all_ids = ex2_pre_ids | ex2_post_ids
                ex2_missing = ex2_all_ids - existing_items
                assert len(ex2_missing) == 0, (
                    f"{len(ex2_missing)} cell ids from CAVE view not found in dataitem/."
                )

                rows_ex2 = []
                for row in conn_view_filtered.iter_rows(named=True):
                    pre = row["pre_nuc_str"]
                    post = row["post_nuc_str"]
                    rows_ex2.append(
                        CellCellConnectivityLong(
                            id=f"{pre}_{post}_{SynapticMeasurementType.SYNAPSE_COUNT.value}",
                            presynaptic_cell=pre,
                            postsynaptic_cell=post,
                            measurement_type=SynapticMeasurementType.SYNAPSE_COUNT.value,
                            modality=Modality.ELECTRON_MICROSCOPY.value,
                            value=float(row[count_col]),
                            unit=Unit.COUNT.value,
                            project_id=PROJECT_ID,
                        )
                    )
                print(f"CellCellConnectivityLong rows (Example 2): {len(rows_ex2)}")
                example2_success = True

In [ ]:
if example2_success:
    table_ex2 = attach_linkml_metadata(
        models_to_table(rows_ex2, schema=schema_cc),
        linkml_class="CellCellConnectivityLong",
    )

    write_deltalake(
        OUTPUT_ROOT + "cellcellconnectivitylong_proofread_to_proofread/",
        table_ex2,
        mode="overwrite",
        predicate=f"project_id = '{PROJECT_ID}'",
        partition_by=["project_id", "measurement_type"],
    )
    print(f"Written to cellcellconnectivitylong_proofread_to_proofread/: {table_ex2.num_rows} rows")
else:
    print("Example 2 skipped — no data written to cellcellconnectivitylong_proofread_to_proofread/.")

In [ ]:
if example2_success:
    ex2_v = pl.read_delta(
        OUTPUT_ROOT + "cellcellconnectivitylong_proofread_to_proofread/"
    ).filter(pl.col("project_id") == PROJECT_ID)
    print(f"Shape: {ex2_v.shape}")
    print(ex2_v.group_by("measurement_type").len())
    print(ex2_v.head(3))
    assert ex2_v.shape[0] == len(rows_ex2)
    assert ex2_v["measurement_type"].n_unique() == 1
    assert ex2_v["measurement_type"][0] == SynapticMeasurementType.SYNAPSE_COUNT.value
else:
    print("Example 2 verification skipped.")

---
## Summary

| Path | Rows |
|------|------|
| `dataset/` | +1 (`minnie65_v1412_proofread` = proofread ∩ CSM cells) |
| `dataitem_dataset_association/` | one per proofread ∩ CSM cell |
| `cellcellconnectivitylong_proofread_pre_to_csm_post/` | 2 × filtered pairs: (proofread ∩ CSM)-pre × CSM-post |
| `cellcellconnectivitylong_proofread_to_proofread/` | 1 × filtered pairs: (proofread ∩ CSM) × (proofread ∩ CSM) — **if CAVE view succeeded** |

Example 2 status is printed above. The proofread ∩ CSM set defines cells with both verified axon reconstructions and CSM dendrite classifications.